# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [ ]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append(".")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface,fcc111
from ase.visualize import view

import site_analysis as sa

In [2]:
# USER INPUT
name = 'pd332.xyz'
n_layers = 10
atoms = surface('Pd',(3,3,2),n_layers)
atoms.center(vacuum=10, axis=2)
adsorbate_height = 2.
site_bond_cutoff = 3.

In [3]:
#visualize slab
write(name, atoms)
view(atoms, viewer='x3d')

In [4]:

# Load slab
slab = read(name)
surface = CustomSurface(slab, n_layers=n_layers)
nslab = len(slab)


In [19]:
slab=read('/home/shikim/pynta/pynta/preprocessing/custom_surfaces/pt_slab.xyz')

In [28]:

#slabrat = slab.copy()
#slabrat.rattle(stdev=0.3)
slab=read('/home/shikim/pynta/pynta/preprocessing/custom_surfaces/pt_slab.xyz')
# Generate symmetry-unique site geometries
#cas = SlabAdsorptionSites(slab, surface='fcc332', composition_effect=True)  # ACAT has surface, custom does not find them all!
cas = SlabAdsorptionSites(slab, surface='fcc111')  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slabrat, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!
site_adjacency = cas.get_neighbor_site_list()

single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

print(f'There are {len(single_sites_lists)} unique sites out of {len(cas.get_sites())}.')
#print(cas.get_sites())
cas.get_unique_sites()
print(f'There are {len(cas.get_unique_sites())} unique sites out of {len(cas.get_sites())}.')
traj = Trajectory("unique_sites.traj", "w")
for g in single_geoms:
    traj.write(g)
traj.close()


There are 8 unique sites out of 54.
There are 4 unique sites out of 54.


In [32]:
import json

def replace_nulls_with_string(data):
    """Recursively replace None with the string 'null' in the data."""
    if isinstance(data, dict):
        return {key: replace_nulls_with_string(value) for key, value in data.items()}
    elif isinstance(data, list):
        return [replace_nulls_with_string(item) for item in data]
    elif data is None:  # Check for None (Python's equivalent of JSON null)
        return "null"  # Replace with the string "null"
    else:
        return data  # Return the data as is if it's not None

# Load the JSON data from the file
with open('sites.json', 'r') as file:
    sites_data = json.load(file)

# Replace None with "null"
modified_sites_data = replace_nulls_with_string(sites_data)

# Write the modified data back to the JSON file
with open('sites.json', 'w') as file:
    json.dump(modified_sites_data, file, indent=4)

print("Replaced nulls with 'null' in sites.json")

Replaced nulls with 'null' in sites.json


In [35]:
import json
import yaml

# Specify the input JSON file and output YAML file
input_json_file = 'sites.json'  # Change this to your JSON file path
output_yaml_file = 'sites.yaml'   # Change this to your desired YAML file path

# Read the JSON file
with open(input_json_file, 'r') as json_file:
    data = json.load(json_file)  # Load JSON data

# Write the data to a YAML file
with open(output_yaml_file, 'w') as yaml_file:
    yaml.dump(data, yaml_file, default_flow_style=False)  # Write YAML data

print(f"Converted '{input_json_file}' to '{output_yaml_file}'")

Converted 'sites.json' to 'sites.yaml'


In [30]:
[cas.get_sites()[i]['morphology'] for i in range(len(cas.get_sites()))]
print(cas.get_unique_sites())
print(cas.get_neighbor_site_list())
with open('site.txt',"a") as f:
    f.write(str(cas.get_sites()))
    f.write("\n")

[{'site': 'ontop', 'surface': 'fcc111', 'morphology': 'terrace', 'position': array([ 4.24644217,  7.35503388, 14.9349216 ]), 'normal': array([6.4e-07, 6.5e-07, 1.0e+00]), 'indices': (27,), 'composition': None, 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27])}, {'site': 'bridge', 'surface': 'fcc111', 'morphology': 'terrace', 'position': array([ 5.66191132,  7.35500668, 14.93494959]), 'normal': array([-4.69e-06, -7.55e-06,  1.00e+00]), 'indices': (27, 28), 'composition': None, 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27, 28])}, {'site': 'fcc', 'surface': 'fcc111', 'morphology': 'terrace', 'position': array([ 1.41544439,  0.8172102 , 14.93494283]), 'normal': array([-1.977e-05,  8.270e-06,  1.000e+00]), 'indices': (27, 28, 30), 'composition': None, 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27, 28, 30])}, {'site': 'hcp', 'surface': 'fcc111', 'morphology': 'terrace', 'positi

In [36]:
import yaml
import numpy as np

def convert_numpy_types(data):
    """Recursively convert NumPy types to native Python types."""
    if isinstance(data, dict):
        return {key: convert_numpy_types(value) for key, value in data.items()}
    elif isinstance(data, list):
        return [convert_numpy_types(item) for item in data]
    elif isinstance(data, np.ndarray):
        return data.tolist()  # Convert NumPy array to list
    elif isinstance(data, (np.int64, np.float64)):  # Check for NumPy numeric types
        return data.item()  # Convert to native Python int or float
    else:
        return data  # Return the data as is if it's already a native type

# Get the neighbor site list
neighbor_site_list = cas.get_neighbor_site_list()

# Convert NumPy types to native Python types
neighbor_site_list = convert_numpy_types(neighbor_site_list)

# Specify the filename for the YAML output
output_filename = 'neighbor_site_list.yaml'

# Save the neighbor site list to a YAML file
with open(output_filename, 'w') as f:
    yaml.dump(neighbor_site_list, f, default_flow_style=False)  # Use block style for pretty printing

print(f"Neighbor site list saved to '{output_filename}'")

Neighbor site list saved to 'neighbor_site_list.yaml'


In [ ]:

# Extract 3-fold site graphs
admols, threefold_geom_indices = sa.classify_threefold_sites(
    single_geoms, single_sites_lists
)


In [ ]:

# Graph isomorphism clustering
iso_mat, clusters = sa.cluster_isomorphic_graphs(admols)


In [ ]:

# Update 3-fold site labels
type_map = sa.update_threefold_site_labels(
    single_sites_lists,
    clusters,
    threefold_geom_indices
)


In [ ]:

# Write 3-fold-only trajectory
traj3 = Trajectory("threefold_sites.traj", "w")
for i in threefold_geom_indices:
    traj3.write(single_geoms[i])
traj3.close()


In [ ]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = False
0 vs 3 = False
0 vs 4 = False
0 vs 5 = False
0 vs 6 = False
0 vs 7 = False
0 vs 8 = False
0 vs 9 = False
0 vs 10 = False
0 vs 11 = False
0 vs 12 = False
0 vs 13 = False
0 vs 14 = False
0 vs 15 = False
0 vs 16 = False
0 vs 17 = False
0 vs 18 = False
0 vs 19 = False
0 vs 20 = False
0 vs 21 = False
0 vs 22 = False
0 vs 23 = False
0 vs 24 = False
0 vs 25 = False
0 vs 26 = False
0 vs 27 = False
0 vs 28 = False
0 vs 29 = False
0 vs 30 = False
0 vs 31 = False
0 vs 32 = False
0 vs 33 = False
0 vs 34 = False
0 vs 35 = False
0 vs 36 = False
0 vs 37 = False
0 vs 38 = False
0 vs 39 = False
0 vs 40 = False
0 vs 41 = False
0 vs 42 = False
0 vs 43 = False
0 vs 44 = False
0 vs 45 = False
0 vs 46 = False
0 vs 47 = False
0 vs 48 = False
0 vs 49 = False
0 vs 50 = False
0 vs 51 = False
0 vs 52 = False
0 vs 53 = False
0 vs 54 = False
0 vs 55 = False
0 vs 56 = False
0 vs 57 = False
0 vs 58 = False
0 vs 59 = False
0 vs 60 = False
0 vs 61 = False
0 vs

In [ ]:

# Distinct 3-fold site types (PRINT)
print("Number of distinct site types:", len(clusters))
for members in clusters.values():
    print("3-fold site type:", members)


Number of distinct 3-fold site types: 36
3-fold site type: [0]
3-fold site type: [1, 13, 14, 16, 19, 20, 21]
3-fold site type: [2]
3-fold site type: [3, 11]
3-fold site type: [4, 23, 24]
3-fold site type: [5]
3-fold site type: [6]
3-fold site type: [7, 8, 10]
3-fold site type: [9, 12]
3-fold site type: [15]
3-fold site type: [17, 25]
3-fold site type: [18, 22]
3-fold site type: [26, 27]
3-fold site type: [28, 29]
3-fold site type: [30]
3-fold site type: [31]
3-fold site type: [32, 33]
3-fold site type: [34]
3-fold site type: [35, 37, 39, 46, 51, 55]
3-fold site type: [36]
3-fold site type: [38]
3-fold site type: [40, 48]
3-fold site type: [41, 42, 43]
3-fold site type: [44, 47]
3-fold site type: [45, 53, 54]
3-fold site type: [49]
3-fold site type: [50]
3-fold site type: [52]
3-fold site type: [56]
3-fold site type: [57, 64, 65, 68]
3-fold site type: [58]
3-fold site type: [59, 67]
3-fold site type: [60, 69, 70]
3-fold site type: [61, 62, 63]
3-fold site type: [66]
3-fold site type: [7

In [ ]:

# Updated 3-fold site labels (PRINT)
print("Updated 3-fold site labels per geometry:")
for geom_idx, label in type_map.items():
    print(f"Geometry {geom_idx} -> {label}")


Updated 3-fold site labels per geometry:
Geometry 0 -> 3fold1
Geometry 1 -> 3fold2
Geometry 13 -> 3fold2
Geometry 14 -> 3fold2
Geometry 16 -> 3fold2
Geometry 19 -> 3fold2
Geometry 20 -> 3fold2
Geometry 21 -> 3fold2
Geometry 2 -> 3fold3
Geometry 3 -> 3fold4
Geometry 11 -> 3fold4
Geometry 4 -> 3fold5
Geometry 23 -> 3fold5
Geometry 24 -> 3fold5
Geometry 5 -> 3fold6
Geometry 6 -> 3fold7
Geometry 7 -> 3fold8
Geometry 8 -> 3fold8
Geometry 10 -> 3fold8
Geometry 9 -> 3fold9
Geometry 12 -> 3fold9
Geometry 15 -> 3fold10
Geometry 17 -> 3fold11
Geometry 25 -> 3fold11
Geometry 18 -> 3fold12
Geometry 22 -> 3fold12
Geometry 26 -> 3fold13
Geometry 27 -> 3fold13
Geometry 28 -> 3fold14
Geometry 29 -> 3fold14
Geometry 30 -> 3fold15
Geometry 31 -> 3fold16
Geometry 32 -> 3fold17
Geometry 33 -> 3fold17
Geometry 34 -> 3fold18
Geometry 35 -> 3fold19
Geometry 37 -> 3fold19
Geometry 39 -> 3fold19
Geometry 46 -> 3fold19
Geometry 51 -> 3fold19
Geometry 55 -> 3fold19
Geometry 36 -> 3fold20
Geometry 38 -> 3fold21
G

In [ ]:
# Site yaml file generated
print("All sites for the custom surfaces are saved in site.yaml")
sa.write_sites_yaml(single_sites_lists, clusters)


All sites for the custom surfaces are saved in site.yaml


In [ ]:
from ase.build import bulk, surface

# Pd bulk (fcc). You can omit 'a' to use ASE's default, or set your preferred lattice constant.
pd_bulk = bulk('Pd', 'fcc', a=3.89)

# Build Pd(332) slab with 5 layers and (e.g.) 12 Å vacuum along z
slab = surface(pd_bulk, (3, 3, 2), layers=5, vacuum=12.0)

# (Optional) make a larger surface cell in x/y
# slab = slab.repeat((2, 2, 1))
write('pd332_surface.xyz',slab)
print(slab)
print("Cell:\n", slab.cell)

Atoms(symbols='Pd5', pbc=[True, True, False], cell=[[8.698304432474183, 0.0, 0.0], [-6.958643545979346, 2.130640748695095, 0.0], [0.0, -0.0, 27.176171699808854]])
Cell:
 Cell([[8.698304432474183, 0.0, 0.0], [-6.958643545979346, 2.130640748695095, 0.0], [0.0, -0.0, 27.176171699808854]])
